# The parser function

The parser will take the raw JSON files and produce dictionaries, rows, clean data, .csv files of information that we need. Then, the data will be saved in the data/processed folder

In [19]:
import os
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

In [20]:
import json
import requests

from datetime import datetime, timedelta, timezone
from pprint import pprint

import csv
import pandas as pd


In [21]:
from config import RAW_DATA_DIR, PROCESSED_DATA_DIR, STATIONS

Let's read in one of the .json files and print it:

In [ ]:
with open(f'{RAW_DATA_DIR}/genoa_principe_departures_2026_05_13T09_13_to_2026_05_13T17_13.json', 'r') as f:
    data = json.load(f)
    pprint(data)

The most important arrays are:
1) data['rides'][n]['id']  
2) data['rides'][n]['status']. \

From there, data['rides'][n]['status']['deviation']['deviation_seconds'] is important for the deviation. \
General information are available at data['rides'][n]['line'] (for example 'direction' or 'code' or 'name' given the initial stop - final stop info). Here, n is the number of _rides_ that are in the .json file. In principle it's okay when data overlap. We need anyway the information of the deviations for different timestamps and stuff. 

In [ ]:
data['rides'][2]['line']

In [ ]:
print(len(data['rides']))

In [ ]:
data['rides'][0]['status']

In [22]:
def extract_dep_delay_id(data, n): # Extract data for the ID of the bus, general information, delay (if given), when it was updated, and the scheduled departure time

    subdata1 = data['rides'][n]['line'] # This is the general information about the bus, such as the code, direction and name of the line
    subdata2 = data['rides'][n]['status']['deviation'] if data['rides'][n]['status']['deviation'] is not None else {'deviation_seconds': None, 'updated_at': None} # For some buses there is no information about the delay


    dict_data = {
        'ride_id': data['rides'][n]['id'],
        'bus_code': subdata1['code'],
        'direction': subdata1['direction'],
        'name': subdata1['name'],
        'deviation_seconds': subdata2['deviation_seconds'],
        'updated_at': subdata2['updated_at'],
        'scheduled_timestamp': data['rides'][n]['status']['scheduled_timestamp'],
        'snapshot_time': data['snapshot_time']
        }
    
 
    return dict_data

In [23]:
def extract_dep_delay_id_total(data): # Extract data for all the rides in the JSON file

    list_of_dicts = []

    for i in range(len(data['rides'])):
        list_of_dicts.append(extract_dep_delay_id(data, i))

    return list_of_dicts

In [ ]:
extract_dep_delay_id(data, 3)

## A slightly better design of the code

Chat suggests to go through the JSON files first, extracting all the data, and keep these in a big row. Then, put this big as a DataFrame. Okay. 
Not to do like a DataFrame for each of the JSON files, and then appending them to the final dataframe.

In [17]:
STATIONS.keys()

dict_keys(['genoa_principe', 'pisa_pietrasantina', 'trieste_autostazione', 'milan_lampugnano', 'napoli_centrale'])

In [24]:
#file_exists = Path(f'{PROCESSED_DATA_DIR}/genoa_principe_departures.csv').exists() # Boolean to check if the file already exists, if not we will create it and write the header, otherwise we will append to it without writing the header


json_files_genoa = sorted(RAW_DATA_DIR.glob('genoa_principe_departures_*.json')) # Get all the JSON files for the departures from Genoa Principe, sorted by name (which includes the timestamp)

json_files_milan = sorted(RAW_DATA_DIR.glob('milan_lampugnano_departures_*.json')) 

for json_files in [json_files_genoa, json_files_milan]: # Loop through the two lists of JSON files (for Genoa and Milan)

    all_rows = []

    name = 'genoa_principe' if json_files == json_files_genoa else 'milan_lampugnano' # Get the name of the station based on which list of JSON files we are currently looping through

    for file in json_files:

        with open(file, 'r') as f:
            data = json.load(f)
            #pprint(data) # just for debugging
    
        all_rows.extend(extract_dep_delay_id_total(data)) # Extend the list of all rows with the data extracted from the current JSON file

    # Once the looping of the files is finished, write a DF for each station

    df = pd.DataFrame(all_rows) # Create a DataFrame from the list of all rows
    df.to_csv(
        f'{PROCESSED_DATA_DIR}/{name}_departures.csv',
        index=False
    )

Btw, we will rebuild the CSV from scratch with addition to new files. It doesn't cost effectively any time, so we can do it. Otherwise, we could track which .json files we added, then have in df.to_csv(mode = 'a') the append mode on. Also, header=not file_exists can help in that case.


In [14]:
df

,id,bus_code,direction,name,deviation_seconds,updated_at,scheduled_timestamp
0,23c8dea6-ab2f-429c-ba95-545b24835042,N456,"Lublin, Bus Station Lublin",Lublin - Milan - Marseille,2859.0,2026-05-20T16:12:41Z,2026-05-20T16:35:00Z
1,279429f3-5d1a-4c7b-a7f8-b2d520720fc8,518,Como,Naples - Milan - Como,0.0,2026-05-20T04:48:55Z,2026-05-20T17:25:00Z
2,f54c0343-266b-4ebc-804e-0228733d2e5a,423,Turin (Vittorio Emanuele),Turin - Ferrara,263.0,2026-05-20T16:18:27Z,2026-05-20T17:30:00Z
3,afeed613-2e47-4b8a-865d-ea644648ff3d,N836,Amsterdam Sloterdijk,Amsterdam - Brussels - Luxemburg - Milan,NaN,NaN,2026-05-20T17:40:00Z
4,7f008d8a-46b9-4a06-9fa6-4a01f7294883,CIT3197,Milan (Malpensa Airport Terminal 1),Standard,NaN,NaN,2026-05-20T17:45:00Z
...,...,...,...,...,...,...,...
4151,562203f0-bd56-4ac4-a4eb-c1ced46b6385,N436,Lecce,Zurich - Lecce,NaN,NaN,2026-05-21T18:25:00Z
4152,abd3a775-9582-4351-a100-2993487c4dc8,N487,Amsterdam Sloterdijk,Amsterdam - Milan,NaN,NaN,2026-05-21T18:25:00Z
4153,8c818f86-3898-4f6c-97fe-c1929cd964ef,472,Verona (Porta Nuova),Verona - Geneve,0.0,2026-05-21T11:59:26Z,2026-05-21T18:30:00Z
4154,b15a5567-3b70-43a7-ab0a-ee4954b77f87,N522,Rome Fiumicino Airport,Milan - Bergamo - Rome,NaN,NaN,2026-05-21T18:30:00Z


In [ ]:
df.head()